In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import itertools
from sklearn.metrics.cluster import adjusted_mutual_info_score, adjusted_rand_score
from scipy import stats
from scipy.stats import pearsonr, kruskal, chi2_contingency, hmean
import seaborn as sns
from ptitprince import PtitPrince as pt
from lifelines import KaplanMeierFitter
from lifelines.statistics import multivariate_logrank_test
from pandas.api.types import is_numeric_dtype
import statsmodels
from statsmodels.stats.multitest import multipletests

In [ ]:
from general_functions import remove_small_clusters, clinical_enrichment, calculate_stability_metrics, add_normalised_metric

# First benchmark: obtaining the best view combination

The aim of this benchmark is to find the best combination of omic views that will give the best clusters. Six omic views from TCGA PDAC database will be used: protein expression arrays (RPPA), RNA-sequence expression (RNAseq), micro-RNA expression (miRNA), mutations binary data (mutations; 1 for at least one mutation present in gene, 0 for no mutations present), methylation, and somatic copy number variations (CNA) in GISTIC2.0 format (-2, -1, 0, 1, 2). All the possible combinations between these data types, containing at least two views were considered, resulting in $2^6 -1 -1 = 57$ combinations. 

This benchmark will also be used to analyse some hyperparameters, such as the number of clusters (2 vs. 3), the data types present, the number of data types and the algorithms used in each experiment. 

In [ ]:
results1_file = pd.read_csv('benchmarking_files/first_bench_file.csv',
                            dtype={'view_combination': str},
                            converters={'y_pred': eval, 'y_pred_idx': eval, 
                                        'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
clinical_data_file = pd.read_csv('raw data/cancer_data_PAAD_clinical_data.tsv', sep='\t', header=0)

In [ ]:
clinical_data_file = pd.read_csv('raw data/cancer_data_PAAD_clinical_data.tsv', sep='\t', header=0)
bench1_2clusters = pd.read_csv('benchmarks_new/firstbench_2clusters.csv',
                               dtype={'view_combination': str},
                               converters={'y_pred': eval, 'y_pred_idx': eval, 
                                           'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
bench1_3clusters = pd.read_csv('benchmarks_new/firstbench_3clusters.csv',
                               dtype={'view_combination': str},
                               converters={'y_pred': eval, 'y_pred_idx': eval, 
                                           'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
bench1_4clusters = pd.read_csv('benchmarks_new/firstbench_4clusters.csv',
                               dtype={'view_combination': str},
                               converters={'y_pred': eval, 'y_pred_idx': eval, 
                                           'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
bench1_5clusters = pd.read_csv('benchmarks_new/firstbench_5clusters.csv',
                               dtype={'view_combination': str},
                               converters={'y_pred': eval, 'y_pred_idx': eval, 
                                           'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
frames = [bench1_2clusters, bench1_3clusters, bench1_4clusters, bench1_5clusters]
results1_file = pd.concat(frames)

In [ ]:
import warnings
warnings.filterwarnings("ignore")
valid_results1, outlier_results1, outlier_patients1 = remove_small_clusters(results1_file, 20, verbose = False)
results_clin1 = clinical_enrichment(valid_results1, clinical_data_file)

In [ ]:
normalised_silhouette1 = add_normalised_metric(results_clin1, variable_to_normalise='view_combination', metric='silhouette', greater_is_better=True)
stability_metrics_results1 = calculate_stability_metrics(normalised_silhouette1, random_state=42, progress_bar=True)
stability_metrics_results1[['AMI', 'ARI']] = stability_metrics_results1[['AMI','ARI']].fillna(value=0)
stability_metrics_results1[['AMI', 'ARI']] = stability_metrics_results1[['AMI', 'ARI']].clip(lower=0)
normalised_ami1 = add_normalised_metric(stability_metrics_results1, variable_to_normalise='view_combination', metric='AMI', greater_is_better=True)

In [ ]:
results1 = normalised_ami1.copy()
results1['n_views'] = normalised_ami1['view_combination'].str.count('1')
# Creating a combined metric using silhouette score (quality of clusters) and AMI (stability metric)
results1['combined_metric'] = results1[['normalised_silhouette', 'normalised_AMI']].apply(lambda x: hmean(x), axis=1)
results1.sort_values('combined_metric', ascending=False, inplace=True)
results1

In [ ]:
sns.set_theme(style='ticks')

In [ ]:
metric = 'combined_metric'
views = ['CNA', 'Methyl', 'Mutations', 'RNAseq', 'RPPA', 'miRNA']

combinations_average = results1.groupby(['view_combination']).mean(numeric_only=True).sort_values(by=metric, ascending=False)
combinations_sorted = combinations_average.index.tolist()

fig, ax = plt.subplots(4, 1, sharex=True, figsize=(16,7), height_ratios=[0.5, 0.3, 0.1, 0.1])

# First plot: boxplots with combined metric score for each combination
sns.boxplot(data=results1, x='view_combination', y=metric, ax=ax[0], 
            order=combinations_sorted, width=0.7, color='white', showmeans=True)
ax[0].set_ylabel('HMean(Adj. Mutual Info., Silhouette)')
ax[0].set_xlabel('')
ax[0].set_axisbelow(True)
ax[0].set_ylim(-0.05, 1.05)
for line in ax[0].lines:
    line.set_color('black')
    line.set_xdata(line.get_xdata() + 0.5)
for patch in ax[0].patches:
    patch.set_edgecolor('black')
    vertices = patch.get_path().vertices
    vertices[:, 0] += 0.5

# Second plot: heatmap showing modalities present 
views_matrix = combinations_average.reset_index()
views_matrix_expanded = views_matrix['view_combination'].apply(lambda x: pd.Series(list(x))).astype(int)
views_matrix_expanded.columns = views
views_matrix_expanded.index = views_matrix['view_combination']
views_ordered = ['Methyl', 'miRNA', 'RNAseq', 'CNA', 'RPPA', 'Mutations'] 
# (^this is a bit of cheating, it is the order from highest to lowest of the modalities as calculated in the next step)
views_matrix_expanded = views_matrix_expanded.reindex(columns=views_ordered)
sns.heatmap(views_matrix_expanded.T, cmap='Blues', linewidths=0.1, linecolor='black', 
            cbar=False, ax=ax[1]).set(xlabel=None)
ax[1].set_ylabel('Modalities')
ax[1].tick_params(axis='x', bottom=True, labelbottom=False)

# Third plot: heatmap showing number of modalities
unique_nviews_data = combinations_average['n_views'].to_frame()
sns.heatmap(unique_nviews_data.T, cmap='Reds', linewidths=0.1, linecolor='black', square=True,
            cbar=True, cbar_kws=dict(use_gridspec=True, location="bottom", pad=0.2), ax=ax[2], 
            yticklabels='', xticklabels=unique_nviews_data.index).set(xlabel=None)
ax[2].set_ylabel('Number of \nmodalities', labelpad=30, rotation=0, va='center')

# Fourth plot: heatmap showing number of features
features = [2185, 385, 1419, 52, 192, 71]
feature_sums = {}
for combination in combinations_sorted:
    total = sum(features[i] for i, bit in enumerate(combination) if bit == '1')
    feature_sums[combination] = total
features_df = pd.DataFrame(feature_sums, index=['number of features'])
sns.heatmap(features_df, cmap='Oranges', linewidths=0.1, linecolor='black', cbar=True, 
            cbar_kws=dict(use_gridspec=True, location="bottom", ticks=[123, 2150, 4304], pad=0.2), ax=ax[3], square=True,
            yticklabels='', xticklabels=features_df.columns).set(xlabel=None)
ax[3].set_ylabel('Number of \nfeatures', labelpad=30, rotation=0, va='center')
ax[3].set_xticklabels('')

plt.tight_layout()
plt.savefig('figures/first_bench_figures/best_combination_combined.svg', bbox_inches='tight')
plt.show()

In [ ]:
metric = 'normalised_silhouette'
views = ['CNA', 'Methyl', 'Mutations', 'RNAseq', 'RPPA', 'miRNA']

combinations_average = results1.groupby(['view_combination']).mean(numeric_only=True).sort_values(by=metric, ascending=False)
combinations_sorted = combinations_average.index.tolist()

fig, ax = plt.subplots(4, 1, sharex=True, figsize=(16,7), height_ratios=[0.5, 0.3, 0.1, 0.1])

# First plot: boxplots with combined metric score for each combination
sns.boxplot(data=results1, x='view_combination', y=metric, ax=ax[0], 
            order=combinations_sorted, width=0.7, color='white', showmeans=True)
ax[0].set_ylabel('Silhouette score (norm.)')
ax[0].set_xlabel('')
ax[0].set_axisbelow(True)
ax[0].set_ylim(-0.1, 1.1)
for line in ax[0].lines:
    line.set_color('black')
    line.set_xdata(line.get_xdata() + 0.5)
for patch in ax[0].patches:
    patch.set_edgecolor('black')
    vertices = patch.get_path().vertices
    vertices[:, 0] += 0.5

# Second plot: heatmap showing modalities present 
views_matrix = combinations_average.reset_index()
views_matrix_expanded = views_matrix['view_combination'].apply(lambda x: pd.Series(list(x))).astype(int)
views_matrix_expanded.columns = views
views_matrix_expanded.index = views_matrix['view_combination']
views_ordered = ['Methyl', 'miRNA', 'RNAseq', 'CNA', 'Mutations', 'RPPA'] 
# (^this is a bit of cheating, it is the order from highest to lowest of the modalities as calculated in the next step)
views_matrix_expanded = views_matrix_expanded.reindex(columns=views_ordered)
sns.heatmap(views_matrix_expanded.T, cmap='Blues', linewidths=0.1, linecolor='black', 
            cbar=False, ax=ax[1]).set(xlabel=None)
ax[1].set_ylabel('Modalities')
ax[1].tick_params(axis='x', bottom=True, labelbottom=False)

# Third plot: heatmap showing number of modalities
unique_nviews_data = combinations_average['n_views'].to_frame()
sns.heatmap(unique_nviews_data.T, cmap='Reds', linewidths=0.1, linecolor='black', square=True,
            cbar=True, cbar_kws=dict(use_gridspec=True, location="bottom", pad=0.2), ax=ax[2], 
            yticklabels='', xticklabels=unique_nviews_data.index).set(xlabel=None)
ax[2].set_ylabel('Number of \nmodalities', labelpad=30, rotation=0, va='center')

# Fourth plot: heatmap showing number of features
features = [2185, 385, 1419, 52, 71, 192]
feature_sums = {}
for combination in combinations_sorted:
    total = sum(features[i] for i, bit in enumerate(combination) if bit == '1')
    feature_sums[combination] = total
features_df = pd.DataFrame(feature_sums, index=['number of features'])
sns.heatmap(features_df, cmap='Oranges', linewidths=0.1, linecolor='black', cbar=True, 
            cbar_kws=dict(use_gridspec=True, location="bottom", ticks=[123, 2150, 4304], pad=0.2), ax=ax[3], square=True,
            yticklabels='', xticklabels=features_df.columns).set(xlabel=None)
ax[3].set_ylabel('Number of \nfeatures', labelpad=30, rotation=0, va='center')
ax[3].set_xticklabels('')

plt.tight_layout()
plt.savefig('figures/first_bench_figures/best_combination_silhouette.svg', bbox_inches='tight')
plt.show()

In [ ]:
metric = 'normalised_AMI'
views = ['CNA', 'Methyl', 'Mutations', 'RNAseq', 'RPPA', 'miRNA']

combinations_average = results1.groupby(['view_combination']).mean(numeric_only=True).sort_values(by=metric, ascending=False)
combinations_sorted = combinations_average.index.tolist()

fig, ax = plt.subplots(4, 1, sharex=True, figsize=(16,7), height_ratios=[0.5, 0.3, 0.1, 0.1])

# First plot: boxplots with combined metric score for each combination
sns.boxplot(data=results1, x='view_combination', y=metric, ax=ax[0], 
            order=combinations_sorted, width=0.7, color='white', showmeans=True)
ax[0].set_ylabel('Adj. Mutual Information (norm.)')
ax[0].set_xlabel('')
ax[0].set_axisbelow(True)
ax[0].set_ylim(-0.1, 1.1)
for line in ax[0].lines:
    line.set_color('black')
    line.set_xdata(line.get_xdata() + 0.5)
for patch in ax[0].patches:
    patch.set_edgecolor('black')
    vertices = patch.get_path().vertices
    vertices[:, 0] += 0.5

# Second plot: heatmap showing modalities present 
views_matrix = combinations_average.reset_index()
views_matrix_expanded = views_matrix['view_combination'].apply(lambda x: pd.Series(list(x))).astype(int)
views_matrix_expanded.columns = views
views_matrix_expanded.index = views_matrix['view_combination']
views_ordered = ['Methyl', 'miRNA', 'RNAseq', 'CNA', 'RPPA', 'Mutation'] 
# (^this is a bit of cheating, it is the order from highest to lowest of the modalities as calculated in the next step)
views_matrix_expanded = views_matrix_expanded.reindex(columns=views_ordered)
sns.heatmap(views_matrix_expanded.T, cmap='Blues', linewidths=0.1, linecolor='black', 
            cbar=False, ax=ax[1]).set(xlabel=None)
ax[1].set_ylabel('Modalities')
ax[1].tick_params(axis='x', bottom=True, labelbottom=False)

# Third plot: heatmap showing number of modalities
unique_nviews_data = combinations_average['n_views'].to_frame()
sns.heatmap(unique_nviews_data.T, cmap='Reds', linewidths=0.1, linecolor='black', square=True,
            cbar=True, cbar_kws=dict(use_gridspec=True, location="bottom", pad=0.2), ax=ax[2], 
            yticklabels='', xticklabels=unique_nviews_data.index).set(xlabel=None)
ax[2].set_ylabel('Number of \nmodalities', labelpad=30, rotation=0, va='center')

# Fourth plot: heatmap showing number of features
features = [2185, 385, 1419, 52, 192, 71]
feature_sums = {}
for combination in combinations_sorted:
    total = sum(features[i] for i, bit in enumerate(combination) if bit == '1')
    feature_sums[combination] = total
features_df = pd.DataFrame(feature_sums, index=['number of features'])
sns.heatmap(features_df, cmap='Oranges', linewidths=0.1, linecolor='black', cbar=True, 
            cbar_kws=dict(use_gridspec=True, location="bottom", ticks=[123, 2150, 4304], pad=0.2), ax=ax[3], square=True,
            yticklabels='', xticklabels=features_df.columns).set(xlabel=None)
ax[3].set_ylabel('Number of \nfeatures', labelpad=30, rotation=0, va='center')
ax[3].set_xticklabels('')

plt.tight_layout()
plt.savefig('figures/first_bench_figures/best_combination_AMI.svg', bbox_inches='tight')
plt.show()

### Raincloud plots

In [ ]:
# Function to plot boxplots for a specific variable
colorblind_palette = sns.color_palette('colorblind')
def raincloud_plots_variables(df, column_name, metric_name, ax, ylabel):
    mean_values = df.groupby(column_name)[metric_name].mean().sort_values(ascending=False)
    sorted_categories = mean_values.index.tolist()
    metric_subsets = [df[df[column_name] == cat][metric_name].values for cat in sorted_categories]
    pt.RainCloud(x=column_name, y=metric_name, data=df, bw=0.2, palette=[colorblind_palette[0]],
                 width_viol=0.4, ax=ax, orient="v", move=0.2, order=sorted_categories, alpha=0.8)
    means = df.groupby(column_name)[metric_name].mean().loc[sorted_categories]
    sns.scatterplot(x=range(len(sorted_categories)), y=means.values, ax=ax, color=colorblind_palette[2], s=100, marker='^', zorder=10)
    ax.set_xticks(range(len(sorted_categories)))
    ax.set_xticklabels(sorted_categories)
    ax.set_ylabel(ylabel)
    ax.set_ylim(-0.05, 1.05)
    ax.set_axisbelow(True)
    pvalue = kruskal(*metric_subsets).pvalue
    if pvalue >= 0.001:
        pvalue_text = f"p = {pvalue:.3f}"
    else:
        pvalue_text = f"p = {pvalue:.2e}"
    return pvalue, pvalue_text

In [ ]:
normalised_silhouette1_alg = add_normalised_metric(results_clin1, variable_to_normalise='algorithm', metric='silhouette', greater_is_better=True)
stability_metrics_results1_alg = calculate_stability_metrics(normalised_silhouette1_alg, random_state=42, progress_bar=True)
stability_metrics_results1_alg[['AMI', 'ARI']] = stability_metrics_results1_alg[['AMI','ARI']].fillna(value=0)
stability_metrics_results1_alg[['AMI', 'ARI']] = stability_metrics_results1_alg[['AMI', 'ARI']].clip(lower=0)
normalised_ami1_alg = add_normalised_metric(stability_metrics_results1_alg, variable_to_normalise='algorithm', metric='AMI', greater_is_better=True)
results1_alg = normalised_ami1_alg.copy()
results1_alg['n_views'] = results1_alg['view_combination'].str.count('1')
results1_alg['combined_metric'] = results1_alg[['normalised_silhouette', 'normalised_AMI']].mean(axis=1)
results1_alg.sort_values('combined_metric', ascending=False, inplace=True)
results1_alg

In [ ]:
normalised_silhouette1_clusters = add_normalised_metric(results_clin1, variable_to_normalise='n_clusters', metric='silhouette', greater_is_better=True)
stability_metrics_results1_clusters = calculate_stability_metrics(normalised_silhouette1_clusters, random_state=42, progress_bar=True)
stability_metrics_results1_clusters[['AMI', 'ARI']] = stability_metrics_results1_clusters[['AMI','ARI']].fillna(value=0)
stability_metrics_results1_clusters[['AMI', 'ARI']] = stability_metrics_results1_clusters[['AMI', 'ARI']].clip(lower=0)
normalised_ami1_clusters = add_normalised_metric(stability_metrics_results1_clusters, variable_to_normalise='n_clusters', metric='AMI', greater_is_better=True)
results1_clusters = normalised_ami1_clusters.copy()
results1_clusters['n_views'] = results1_clusters['view_combination'].str.count('1')
results1_clusters["normalised_AMI"] = results1_clusters["normalised_AMI"].fillna(value=0)
results1_clusters['combined_metric'] = results1_clusters[['normalised_silhouette', 'normalised_AMI']].mean(axis=1)
results1_clusters.sort_values('combined_metric', ascending=False, inplace=True)
results1_clusters

In [ ]:
from statannotations.Annotator import Annotator
import sys
import contextlib
import io

metric='combined_metric'
ylabel='HMean(Adj. Mutual Info.,\nSilhouette)'
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 1], width_ratios=[0.45, 0.55])

# plot for combinations containing each data type
ax1 = fig.add_subplot(gs[0, :])
plot_data = []
views = ['CNA', 'Methyl', 'Mutations', 'RNAseq', 'RPPA', 'miRNA']
for i, view in enumerate(views):
    subset = results1[results1['view_combination'].str[i] == '1']
    subset['data_type'] = view
    plot_data.append(subset)
combined_df = pd.concat(plot_data)
pvalue, pvalue_text = raincloud_plots_variables(combined_df, 'data_type', metric, ax1, ylabel)
ax1.set_xlabel('Data type')
ax1.legend(title=f"{pvalue_text}", loc=3)
unique_data_types = combined_df['data_type'].unique()
pairs = list(itertools.combinations(unique_data_types, 2))
ns_pairs = []
for pair in pairs:
    group1 = combined_df[combined_df['data_type'] == pair[0]][metric]
    group2 = combined_df[combined_df['data_type'] == pair[1]][metric]
    stat, pval = kruskal(group1, group2)
    if pval > 0.05:
        ns_pairs.append(pair)
annotator = Annotator(ax1, ns_pairs, data=combined_df, x='data_type', y=metric)
annotator.configure(test='Kruskal', text_format='star', loc='inside')
with contextlib.redirect_stdout(io.StringIO()):
    annotator.apply_and_annotate()
# annotator = Annotator(ax1, pairs, data=combined_df, x='data_type', y=metric)
# annotator.configure(test='Kruskal', text_format='star', loc='outside')
# with contextlib.redirect_stdout(io.StringIO()):
#     annotator.apply_and_annotate()

# plot for algorithm
ax2 = fig.add_subplot(gs[1, :])
pvalue, pvalue_text = raincloud_plots_variables(results1_alg, 'algorithm', metric, ax2, ylabel)
ax2.set_xlabel('Algorithm')
ax2.legend(title=f"{pvalue_text}", loc=3)
unique_algs = results1_alg['algorithm'].unique()
pairs = list(itertools.combinations(unique_algs, 2))
ns_pairs = []
for pair in pairs:
    group1 = results1_alg[results1_alg['algorithm'] == pair[0]][metric]
    group2 = results1_alg[results1_alg['algorithm'] == pair[1]][metric]
    stat, pval = kruskal(group1, group2)
    if pval > 0.05:
        ns_pairs.append(pair)
annotator = Annotator(ax2, ns_pairs, data=results1_alg, x='algorithm', y=metric)
annotator.configure(test='Kruskal', text_format='star', loc='inside')
with contextlib.redirect_stdout(io.StringIO()):
    annotator.apply_and_annotate()
# annotator = Annotator(ax2, pairs, data=results1_alg, x='algorithm', y=metric)
# annotator.configure(test='Kruskal', text_format='star', loc='outside')
# with contextlib.redirect_stdout(io.StringIO()):
#     annotator.apply_and_annotate()

# plot for number of clusters
ax3 = fig.add_subplot(gs[2, 0])
pvalue, pvalue_text = raincloud_plots_variables(results1_clusters, 'n_clusters', metric, ax3, ylabel)
ax3.set_xlabel('Number of clusters')
ax3.legend(title=f"{pvalue_text}", loc=3)
unique_clusters = results1_clusters['n_clusters'].unique()
# pairs = list(itertools.combinations(unique_clusters, 2))
# annotator = Annotator(ax3, pairs, data=results1_clusters, x='n_clusters', y=metric)
# annotator.configure(test='Kruskal', text_format='star', loc='outside')
# with contextlib.redirect_stdout(io.StringIO()):
#     annotator.apply_and_annotate()
# ALL SIGNIFICANT

# plot for combinations with a specific number of views
ax4 = fig.add_subplot(gs[2, 1])
pvalue, pvalue_text = raincloud_plots_variables(results1, 'n_views', metric, ax4, ylabel)
ax4.set_xlabel('Number of modalities')
ax4.legend(title=f"{pvalue_text}", loc=3).set_zorder(100)
unique_views = results1['n_views'].unique()
pairs = list(itertools.combinations(unique_views, 2))
sig_pairs = []
for pair in pairs:
    group1 = results1[results1['n_views'] == pair[0]][metric]
    group2 = results1[results1['n_views'] == pair[1]][metric]
    stat, pval = kruskal(group1, group2)
    if pval <= 0.05:
        sig_pairs.append(pair)
annotator = Annotator(ax4, sig_pairs, data=results1, x='n_views', y=metric)
annotator.configure(test='Kruskal', text_format='star', loc='inside')
with contextlib.redirect_stdout(io.StringIO()):
    annotator.apply_and_annotate()
# annotator = Annotator(ax4, pairs, data=results1, x='n_views', y=metric)
# annotator.configure(test='Kruskal', text_format='star', loc='outside')
# with contextlib.redirect_stdout(io.StringIO()):
#     annotator.apply_and_annotate()

fig.subplots_adjust(wspace = 0.3, hspace = 0.4)
plt.savefig('figures/first_bench_figures/raincloud_plots_combined.svg', bbox_inches='tight')
plt.show()

In [ ]:
metric='normalised_silhouette'
ylabel='Silhouette score (norm.)'
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 1], width_ratios=[0.3, 0.7])

# plot for combinations containing each data type
ax1 = fig.add_subplot(gs[0, :])
plot_data = []
views = ['CNA', 'Methyl', 'Mutation', 'RNAseq', 'RPPA', 'miRNA']
for i, view in enumerate(views):
    subset = results1[results1['view_combination'].str[i] == '1']
    subset['data_type'] = view
    plot_data.append(subset)
combined_df = pd.concat(plot_data)
pvalue, pvalue_text = raincloud_plots_variables(combined_df, 'data_type', metric, ax1, ylabel)
ax1.set_xlabel('Data type')
ax1.legend(title=f"{pvalue_text}", loc=3)

# plot for algorithm
ax2 = fig.add_subplot(gs[1, :])
pvalue, pvalue_text = raincloud_plots_variables(results1_alg, 'algorithm', metric, ax2, ylabel)
ax2.set_xlabel('Algorithm')
ax2.legend(title=f"{pvalue_text}", loc=3)

# plot for number of clusters
ax3 = fig.add_subplot(gs[2, 0])
pvalue, pvalue_text = raincloud_plots_variables(results1_clusters, 'n_clusters', metric, ax3, ylabel)
ax3.set_xlabel('Number of clusters')
ax3.legend(title=f"{pvalue_text}", loc=3)

# plot for combinations with a specific number of views
ax4 = fig.add_subplot(gs[2, 1])
pvalue, pvalue_text = raincloud_plots_variables(results1, 'n_views', metric, ax4, ylabel)
ax4.set_xlabel('Number of modalities')
ax4.legend(title=f"{pvalue_text}", loc=3)

fig.subplots_adjust(wspace = 0.3, hspace = 0.4)
plt.savefig('figures/first_bench_figures/raincloud_plots_silhouette.svg', bbox_inches='tight')
plt.show()

In [ ]:
metric='normalised_AMI'
ylabel='Adj. Mutual Information (norm.)'
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 1], width_ratios=[0.3, 0.7])

# plot for combinations containing each data type
ax1 = fig.add_subplot(gs[0, :])
plot_data = []
views = ['CNA', 'Methyl', 'Mutation', 'RNAseq', 'RPPA', 'miRNA']
for i, view in enumerate(views):
    subset = results1[results1['view_combination'].str[i] == '1']
    subset['data_type'] = view
    plot_data.append(subset)
combined_df = pd.concat(plot_data)
pvalue, pvalue_text = raincloud_plots_variables(combined_df, 'data_type', metric, ax1, ylabel)
ax1.set_xlabel('Data type')
ax1.legend(title=f"{pvalue_text}", loc=3).set_zorder(100)

# plot for algorithm
ax2 = fig.add_subplot(gs[1, :])
pvalue, pvalue_text = raincloud_plots_variables(results1_alg, 'algorithm', metric, ax2, ylabel)
ax2.set_xlabel('Algorithm')
ax2.legend(title=f"{pvalue_text}", loc=3)

# plot for number of clusters
ax3 = fig.add_subplot(gs[2, 0])
pvalue, pvalue_text = raincloud_plots_variables(results1_clusters, 'n_clusters', metric, ax3, ylabel)
ax3.set_xlabel('Number of clusters')
ax3.legend(title=f"{pvalue_text}", loc=3)

# plot for combinations with a specific number of views
ax4 = fig.add_subplot(gs[2, 1])
pvalue, pvalue_text = raincloud_plots_variables(results1, 'n_views', metric, ax4, ylabel)
ax4.set_xlabel('Number of modalities')
ax4.legend(title=f"{pvalue_text}", loc=3).set_zorder(100)

fig.subplots_adjust(wspace = 0.3, hspace = 0.3)
plt.savefig('figures/first_bench_figures/raincloud_plots_AMI.svg', bbox_inches='tight')
plt.show()

### Scatter plots

In [ ]:
# Function to create new df for data types used
views = ['CNA', 'Methyl', 'Mutation', 'RNAseq', 'RPPA', 'miRNA']
def expand_views(df):
    expanded_rows = []
    for idx, row in df.iterrows():
        view_combination = row['view_combination']
        for i, bit in enumerate(view_combination):
            if bit == '1':
                new_row = row.copy()
                new_row['views_present'] = views[i]
                expanded_rows.append(new_row)
    expanded_df = pd.DataFrame(expanded_rows)
    return expanded_df

# Scatter plots (mean number of clinically enriched parameters)
def plot_scatterplot(df, column_name, ax):
    summarised_by_variable = df.groupby(column_name, as_index=False).mean(numeric_only=True)
    sns.scatterplot(x='pvalue_logrank', y='n_enriched_clinical', data = summarised_by_variable,
                    hue=column_name, style=column_name, palette = "colorblind", markers=True, s=400, ax=ax, zorder=3)
    ax.set_ylim(-0.05, 1)
    ax.set_ylabel('Mean no. significant clinical parameters')
    ax.set_xlabel('Log-rank test p-value')
    
fig, ax = plt.subplots(1, 4, figsize=(20, 4), sharey=True)
fig.subplots_adjust(wspace = 0.2, hspace = 0.2)
results1_alg_exp = expand_views(results1_alg)
plot_scatterplot(results1_alg_exp, 'algorithm', ax[0])
ax[0].legend(title='Algorithm')
results1_clusters_exp = expand_views(results1_clusters)
plot_scatterplot(results1_clusters_exp, 'n_clusters', ax[1])
ax[1].legend(title='Number\nof clusters')
results1_exp = expand_views(results1)
plot_scatterplot(results1_exp, 'n_views', ax[2])
ax[2].legend(title='Number of \nmodalities')
plot_scatterplot(results1_exp, 'views_present', ax[3])
ax[3].legend(title='Modalities present')

fig.subplots_adjust(wspace = 0.1)
plt.savefig('figures/first_bench_figures/clinical_enrichment.svg', bbox_inches='tight')
plt.show()